SORT Test

In [12]:
import os
import cv2
import json
import numpy as np

In [28]:
folder = "Fully_annotate/Fully_annotate/BLR_1651660052.1477683"
output_folder = "Fully_annotate/Fully_annotate/BLR_1651660052.1477683/output_stationary"
os.makedirs(output_folder, exist_ok=True)

In [29]:
movement_threshold = 5  # pixels
valid_extensions = (".jpg", ".jpeg", ".png")

def polygon_to_bbox(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return [min(xs), min(ys), max(xs), max(ys)]

def get_centroid(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

In [30]:
json_files = sorted([f for f in os.listdir(folder) if f.endswith(".json")])

In [31]:
for i in range(len(json_files) - 1):
    curr_json = os.path.join(folder, json_files[i])
    next_json = os.path.join(folder, json_files[i + 1])
    curr_img_path = os.path.join(folder, json_files[i].replace(".json", ".jpg"))
    img = cv2.imread(curr_img_path)
    if img is None:
        print(f"Skipping: {curr_img_path}")
        continue

    # Load current + next JSON
    with open(curr_json) as f:
        curr_data = json.load(f)
    with open(next_json) as f:
        next_data = json.load(f)

    curr_objs = []
    for s in curr_data["shapes"]:
        label = s["label"]
        bbox = polygon_to_bbox(s["points"])
        curr_objs.append({"label": label, "bbox": bbox})

    next_objs = []
    for s in next_data["shapes"]:
        label = s["label"]
        bbox = polygon_to_bbox(s["points"])
        next_objs.append({"label": label, "bbox": bbox})

    results = []
    for obj in curr_objs:
        cls = obj["label"]
        box = obj["bbox"]
        cx, cy = get_centroid(box)

        # Find closest object of same class in next frame
        min_dist = float("inf")
        for nobj in next_objs:
            if nobj["label"] != cls:
                continue
            nx, ny = get_centroid(nobj["bbox"])
            dist = np.sqrt((cx - nx) ** 2 + (cy - ny) ** 2)
            min_dist = min(min_dist, dist)

        stationary = min_dist < movement_threshold
        color = (0, 0, 255) if stationary else (0, 255, 0)  # red = stationary, green = moving
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{cls} {'S' if stationary else 'M'}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        results.append({
            "class": str(cls),
            "bbox": [float(x) for x in box],
            "stationary": bool(stationary)
        })

    # Save annotated image and result JSON
    save_img_path = os.path.join(output_folder, f"output_{os.path.basename(curr_img_path)}")
    save_json_path = os.path.join(output_folder, f"result_{os.path.basename(curr_json)}")

    cv2.imwrite(save_img_path, img)
    with open(save_json_path, "w") as jf:
        json.dump(results, jf, indent=4)

    print(f"Processed {json_files[i]} → moving/stationary detection complete")

print("All frames processed.")

Processed frame0.json → moving/stationary detection complete
Processed frame10.json → moving/stationary detection complete
Processed frame12.json → moving/stationary detection complete
Processed frame14.json → moving/stationary detection complete
Processed frame16.json → moving/stationary detection complete
Processed frame18.json → moving/stationary detection complete
Processed frame2.json → moving/stationary detection complete
Processed frame20.json → moving/stationary detection complete
Processed frame22.json → moving/stationary detection complete
Processed frame24.json → moving/stationary detection complete
Processed frame26.json → moving/stationary detection complete
Processed frame28.json → moving/stationary detection complete
Processed frame30.json → moving/stationary detection complete
Processed frame32.json → moving/stationary detection complete
Processed frame34.json → moving/stationary detection complete
Processed frame36.json → moving/stationary detection complete
Processed 